# RAG + Agents avec Mistral AI

Notebook de référence — construit from scratch, sans LangChain ni librairie magique.

## Plan
1. Appel LLM basique
2. Embeddings
3. Retrieval (similarité cosinus)
4. RAG complet (retrieval + génération)
5. Agent (tool calling + boucle ReAct)
6. RAG + Agent combiné

## Setup

In [1]:
from mistralai import Mistral
from dotenv import load_dotenv
import os
import json
import numpy as np

load_dotenv()
client = Mistral(api_key=os.getenv("MISTRAL_API_KEY"))
print("✅ Client Mistral initialisé")

✅ Client Mistral initialisé


---
## 1. Appel LLM basique

Le cas le plus simple : on envoie un message, le LLM répond.

Structure d'un message :
- `system` : le comportement global du LLM
- `user` : la question
- `assistant` : la réponse (utilisé pour les conversations multi-tour)

La réponse est dans `response.choices[0].message.content`.

In [2]:
response = client.chat.complete(
    model="mistral-small-latest",
    messages=[
        {"role": "system", "content": "Tu es un assistant utile."},
        {"role": "user", "content": "C'est quoi le RAG en une phrase ?"}
    ]
)

print(response.choices[0].message.content)

Le **RAG** (Retrieval-Augmented Generation) est une technique d'IA qui combine la récupération d'informations pertinentes (retrieval) avec la génération de texte (generation) pour améliorer la précision et la pertinence des réponses.


---
## 2. Embeddings

Un **embedding** = un vecteur de 1024 nombres qui représente le **sens** d'un texte.

```
"Flying Blue est le programme de fidélité" → [0.03, -0.02, 0.87, ...] (1024 valeurs)
"Quel est le programme de fidélité ?"      → [0.03, -0.02, 0.85, ...] (très proche)
"La flotte comprend des Airbus A350"       → [-0.1, 0.4, -0.2, ...]   (éloigné)
```

Deux phrases avec le même sens → vecteurs proches.  
C'est ça qui permet la **recherche sémantique** : trouver les docs pertinents sans matching exact de mots-clés.

In [3]:
# Base de documents Air France
docs = [
    "Air France est la compagnie aérienne nationale française.",
    "Le hub principal d'Air France est l'aéroport Charles de Gaulle.",
    "Air France fait partie du groupe Air France-KLM depuis 2004.",
    "La flotte d'Air France comprend des Airbus A350 et des Boeing 777.",
    "Air France emploie environ 45 000 personnes dans le monde.",
    "Le programme de fidélité d'Air France s'appelle Flying Blue.",
]

# Embed tous les docs en une seule requête API
response = client.embeddings.create(
    model="mistral-embed",
    inputs=docs
)

doc_embeddings = np.array([item.embedding for item in response.data])
print(f"Shape : {doc_embeddings.shape}")  # (6 docs, 1024 dimensions)

Shape : (6, 1024)


---
## 3. Retrieval — Similarité cosinus

Pour trouver les docs pertinents, on mesure l'**angle** entre le vecteur de la question et chaque vecteur de doc.

```
cos_sim = (A · B) / (|A| * |B|)
```

- Résultat entre 0 et 1
- 1 = même direction = même sens
- 0 = perpendiculaires = sens sans rapport

**Pourquoi cosinus et pas distance euclidienne ?**  
La distance euclidienne dépend de la longueur des vecteurs — un texte long aurait un score plus élevé juste parce qu'il est long. Le cosinus mesure uniquement la **direction**, pas la magnitude.

In [4]:
def retrieve(question, top_k=2):
    # Embed la question
    q_response = client.embeddings.create(model="mistral-embed", inputs=[question])
    q_emb = np.array(q_response.data[0].embedding)
    
    # Similarité cosinus : produit scalaire normalisé
    scores = doc_embeddings @ q_emb
    scores = scores / (np.linalg.norm(doc_embeddings, axis=1) * np.linalg.norm(q_emb))
    
    # Top-K : argsort croissant → inverser → prendre les K premiers
    top_indices = np.argsort(scores)[::-1][:top_k]
    return [docs[i] for i in top_indices], scores[top_indices]

# Test
question = "Quel est le programme de fidélité d'Air France ?"
results, scores = retrieve(question)
for doc, score in zip(results, scores):
    print(f"[{score:.3f}] {doc}")

# Flying Blue doit avoir un score nettement plus élevé (~0.90)

[0.907] Le programme de fidélité d'Air France s'appelle Flying Blue.
[0.837] Air France est la compagnie aérienne nationale française.


---
## 4. RAG complet — Retrieval + Génération

On assemble les deux briques précédentes :

```
Question
  → retrieve() → top-2 docs pertinents
  → LLM génère une réponse en se basant UNIQUEMENT sur ces docs
```

Le mot clé dans le system prompt : **"UNIQUEMENT"**.  
Sans ça, le LLM mélange ses connaissances générales avec le contexte → hallucinations possibles.  
Avec ça, si la réponse n'est pas dans les docs, il le dit.

In [5]:
def rag(question, top_k=2):
    # 1. Retrieval
    context_docs, _ = retrieve(question, top_k)
    context = "\n".join(f"- {doc}" for doc in context_docs)
    
    # 2. Génération avec contexte
    response = client.chat.complete(
        model="mistral-small-latest",
        messages=[
            {"role": "system", "content": "Tu réponds UNIQUEMENT en te basant sur le contexte fourni. Si la réponse n'est pas dans le contexte, dis-le."},
            {"role": "user", "content": f"Contexte :\n{context}\n\nQuestion : {question}"}
        ]
    )
    return response.choices[0].message.content

# Test 1 : réponse dans les docs → doit répondre
print(rag("Quel est le programme de fidélité d'Air France ?"))
print("---")
# Test 2 : réponse hors docs → doit dire qu'il ne sait pas, pas halluciner
print(rag("Quel est le PDG d'Air France ?"))

Le programme de fidélité d'Air France s'appelle Flying Blue.
---
D'après le contexte fourni, je ne peux pas répondre à cette question.


---
## 5. Agent — Tool calling + Boucle ReAct

Un LLM classique répond d'un coup. Un **agent** peut agir dans le monde : appeler des APIs, chercher des infos, exécuter du code.

La boucle **ReAct** (Reason + Act) :
```
Question
  → LLM réfléchit : "j'ai besoin de get_weather"
  → on appelle get_weather → résultat ajouté aux messages
  → LLM réfléchit : "j'ai besoin de get_flight_info"
  → on appelle get_flight_info → résultat ajouté aux messages
  → LLM : "j'ai tout" → réponse finale (pas de tool_calls)
```

**Point clé** : les résultats des outils sont ajoutés dans `messages` à chaque tour.  
C'est comme ça que le LLM "voit" ce que les outils ont retourné et peut continuer à raisonner.

In [6]:
# Définition des outils (schéma JSON que Mistral comprend)
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Retourne la météo actuelle d'une ville",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "Nom de la ville"}
                },
                "required": ["city"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_flight_info",
            "description": "Retourne des infos sur une route Air France",
            "parameters": {
                "type": "object",
                "properties": {
                    "route": {"type": "string", "description": "La route ex: Paris-Tokyo"}
                },
                "required": ["route"]
            }
        }
    }
]

# Implémentation des outils (simulée — en prod ce serait de vraies APIs)
def call_tool(name, arguments):
    if isinstance(arguments, str):
        arguments = json.loads(arguments)
    if name == "get_weather":
        return f"Il fait 22°C et ensoleillé à {arguments['city']}."
    elif name == "get_flight_info":
        return f"La route {arguments['route']} opère 2 vols par jour, durée 8h30."

# Boucle ReAct
def run_agent(question, max_turns=5):
    messages = [
        {"role": "system", "content": "Tu es un assistant utile. Utilise les outils disponibles."},
        {"role": "user", "content": question}
    ]
    
    for turn in range(max_turns):
        response = client.chat.complete(
            model="mistral-small-latest",
            messages=messages,
            tools=tools
        )
        message = response.choices[0].message
        
        # Pas de tool_calls → le LLM a tout ce qu'il faut → réponse finale
        if not message.tool_calls:
            print(f"\nRéponse finale ({turn+1} tours) :")
            print(message.content)
            return
        
        # Appel des outils + ajout des résultats dans messages
        for tool_call in message.tool_calls:
            name = tool_call.function.name
            result = call_tool(name, tool_call.function.arguments)
            print(f"Tour {turn+1} — {name} → {result}")
            messages.append({"role": "assistant", "tool_calls": [tool_call]})
            messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})

# Test : 2 outils appelés en parallèle au tour 1
run_agent("Quel temps fait-il à Tokyo et donne moi des infos sur Paris-Tokyo ?")

Tour 1 — get_weather → Il fait 22°C et ensoleillé à Tokyo.
Tour 1 — get_flight_info → La route Paris-Tokyo opère 2 vols par jour, durée 8h30.

Réponse finale (2 tours) :
Voici les informations demandées :

1. **Météo à Tokyo** : Il fait actuellement **22°C** et le temps est **ensoleillé**.

2. **Infos sur la route Paris-Tokyo** :
   - **Nombre de vols par jour** : 2 vols.
   - **Durée du vol** : Environ **8h30**.


---
## 6. RAG + Agent combiné

On ajoute `search_docs` comme outil. C'est notre RAG de la brique 4 emballé dans un outil.

```
search_docs(query)
  → embed(query)
  → cosine similarity avec doc_embeddings
  → retourne les top-2 docs
```

**L'agent décide lui-même quel outil utiliser** — pas de logique `if/else` à coder :
- Question sur Air France → `search_docs`
- Question météo → `get_weather`
- Question mixte → les deux en parallèle

C'est le **pattern de base de tous les assistants RAG en production**.

In [7]:
tools_rag = [
    {
        "type": "function",
        "function": {
            "name": "search_docs",
            "description": "Recherche dans la base de connaissances Air France. Utilise cet outil pour toute question sur Air France.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "La question à rechercher"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "get_weather",
            "description": "Retourne la météo actuelle d'une ville",
            "parameters": {
                "type": "object",
                "properties": {
                    "city": {"type": "string", "description": "Nom de la ville"}
                },
                "required": ["city"]
            }
        }
    }
]

def call_tool_rag(name, arguments):
    if isinstance(arguments, str):
        arguments = json.loads(arguments)
    if name == "search_docs":
        # Notre RAG de la brique 4, emballé comme un outil
        results, _ = retrieve(arguments["query"], top_k=2)
        return "\n".join(f"- {doc}" for doc in results)
    elif name == "get_weather":
        return f"Il fait 22°C et ensoleillé à {arguments['city']}."

def run_rag_agent(question, max_turns=5):
    messages = [
        {"role": "system", "content": "Tu es un assistant expert Air France. Pour toute question sur Air France utilise search_docs. Pour la météo utilise get_weather."},
        {"role": "user", "content": question}
    ]
    
    for turn in range(max_turns):
        response = client.chat.complete(
            model="mistral-small-latest",
            messages=messages,
            tools=tools_rag
        )
        message = response.choices[0].message
        
        if not message.tool_calls:
            print(f"\nRéponse finale ({turn+1} tours) :")
            print(message.content)
            return
        
        for tool_call in message.tool_calls:
            name = tool_call.function.name
            result = call_tool_rag(name, tool_call.function.arguments)
            print(f"Tour {turn+1} — {name}")
            print(f"  → {result[:80]}..." if len(result) > 80 else f"  → {result}")
            messages.append({"role": "assistant", "tool_calls": [tool_call]})
            messages.append({"role": "tool", "content": result, "tool_call_id": tool_call.id})

# Test 1 : question docs → search_docs
print("=== Question documentaire ===")
run_rag_agent("C'est quoi le programme de fidélité Air France ?")

# Test 2 : question météo → get_weather
print("\n=== Question météo ===")
run_rag_agent("Quel temps fait-il à Paris ?")

# Test 3 : question mixte → les deux en parallèle
print("\n=== Question mixte ===")
run_rag_agent("Quel temps fait-il à CDG et c'est quoi la flotte Air France ?")

=== Question documentaire ===
Tour 1 — search_docs
  → - Le programme de fidélité d'Air France s'appelle Flying Blue.
- Air France fait...

Réponse finale (2 tours) :
Le programme de fidélité d'**Air France** s'appelle **Flying Blue**.

C'est le programme de fidélité du groupe **Air France-KLM**, qui permet aux membres de cumuler des miles et de bénéficier d'avantages exclusifs, comme des billets gratuits, des surclassements, ou encore l'accès à des services premium.

Si tu veux plus de détails sur les avantages ou comment rejoindre le programme, fais-le-moi savoir ! 😊

=== Question météo ===
Tour 1 — get_weather
  → Il fait 22°C et ensoleillé à Paris.

Réponse finale (2 tours) :
Il fait actuellement **22°C** et **ensoleillé** à Paris. 😊

=== Question mixte ===
Tour 1 — get_weather
  → Il fait 22°C et ensoleillé à Paris.
Tour 1 — search_docs
  → - La flotte d'Air France comprend des Airbus A350 et des Boeing 777.
- Air Franc...

Réponse finale (2 tours) :
Voici les informations demandé

---
## Récap

| Brique | Ce que ça fait | Ce que ça apporte |
|--------|----------------|-------------------|
| 1. LLM basique | Appel API simple | Base de tout |
| 2. Embeddings | Texte → vecteur 1024D | Représentation sémantique |
| 3. Retrieval | Cosine similarity | Trouver les docs pertinents |
| 4. RAG | Retrieval + génération | Réponses ancrées dans les docs |
| 5. Agent | Tool calling + boucle ReAct | LLM qui agit dans le monde |
| 6. RAG + Agent | search_docs comme outil | Assistant RAG en production |

## Prochaines étapes

- **Hybrid search** : combiner embeddings (sémantique) + BM25 (keyword) → meilleurs résultats
- **Vrai vector store** : remplacer numpy par Chroma ou Weaviate pour scaler
- **Chunking** : découper de vrais documents intelligemment
- **Évaluation RAG** : mesurer recall, precision, faithfulness
- **Streaming** : réponses en temps réel pour la prod